# Project 6: Agentic AI Systems
## DataDesk: A Tool-Using Reasoning Agent for Sales Data Q&A

**Name:** Timothy

## Task 1 — Agentic Task and System Scope

**Goal of the agent:** DataDesk is a scoped question-answering assistant that helps a user explore a small sales dataset for a fictional coffee shop chain. Given a natural-language question (for example, "What were total sales in March?" or "How does that compare to February?"), the agent decides what the question is actually asking, retrieves or computes the answer using its tools, and returns a grounded answer with a visible reasoning trace -- refusing or asking for clarification when a request falls outside what it can responsibly do.

**Single-agent vs. multi-agent:** DataDesk is a **single-agent** system. The task (answering questions about one dataset using two tools) does not require role specialization or delegation between multiple cooperating agents, so a single reasoning loop with tool access is the appropriate scope; this trade-off is discussed further in Task 2.

**Decisions the agent is responsible for:**
- Classifying what kind of request it has received (a data lookup/aggregation, a raw calculation, a follow-up question that depends on conversation memory, an out-of-scope request, or an unclear request)
- Choosing which tool(s) to invoke, and with what parameters, to answer a lookup or calculation request
- Deciding when a follow-up question has enough context (from memory) to be resolved, versus when it must ask for clarification
- Deciding when to refuse a request outright because it is destructive, out of scope, or unsupported by the data

**Actions/tools available to the agent:**
1. **`query_dataset`** — a real pandas-based tool that filters and aggregates the sales dataset (by month, product, category, or region, and by sum/average/count)
2. **`safe_calculate`** — a restricted arithmetic calculator tool (addition, subtraction, multiplication, division, percentages) that never executes arbitrary code
3. **Memory read/write** — not a tool the agent "calls" explicitly, but a component it consults and updates every turn (see Task 2)

**Data source:** `sales_data.csv` is a small, illustrative sales log (4,031 rows: date, month, product, category, region, quantity, unit price, total sales) generated for this project specifically so the agent has a concrete, bounded domain of facts to reason over. It is not used to train or evaluate a statistical model -- it exists purely as the object of the agent's tool-based lookups, so the dataset realism/sourcing rules from the earlier statistics/ML projects do not apply here. Its generation script is included in this repository for transparency.

**Boundaries (what the agent is explicitly *not* allowed to do):** DataDesk cannot modify, delete, or add rows to the dataset (it only reads from it); it cannot answer questions about columns that do not exist in the dataset (for example, employee names or profit margins, which are not tracked); and it cannot run more than a fixed number of reasoning steps per question, after which it must stop and report that it could not resolve the request rather than looping indefinitely.

## Task 2 — Agent Architecture

**Components:**
- **Persona / system role:** DataDesk is scoped as a read-only sales-data analyst assistant. Its persona is defined by what it refuses to do as much as by what it does: it will not fabricate numbers that are not in the data, will not perform destructive actions, and will say "I don't know" rather than guess.
- **Reasoning loop:** a ReAct-style (Reason + Act) loop -- Thought -> Action -> Observation, repeated up to a fixed step limit -- adapted from the pattern introduced by Yao et al. (2023) for combining stepwise reasoning with tool calls. At each step the agent produces a *Thought* (what it currently believes it needs to do), takes an *Action* (call a tool, consult memory, or produce a final answer), and records the *Observation* (the tool's result) before deciding the next step.
- **Memory:** two lightweight components: (1) *working memory* (`last_topic`), which remembers the most recent metric/filter combination the user asked about, so a short follow-up like "how does that compare to February?" can be resolved without the user repeating themselves; and (2) a *session history* (`history`), a list of every question asked and answer given in the session, plus an *answer cache* keyed by normalized query so a repeated question is not recomputed.
- **Tools:** `query_dataset` (pandas-based lookup/aggregation over `sales_data.csv`) and `safe_calculate` (a restricted arithmetic evaluator built on Python's `ast` module rather than `eval`, so it can only ever compute arithmetic and can never execute arbitrary code).
- **Safeguards:** a maximum step count per query (prevents infinite reasoning loops); a scope guard that pattern-matches for destructive or out-of-scope requests (edit/delete requests, or requests about columns the dataset does not have) and refuses them before any tool runs; and a grounding rule that the final answer must be traceable to a tool observation or memory lookup -- if no tool produced a usable result, the agent reports that it cannot answer rather than inventing a number.

**Architecture diagram (component interaction):**

```
                 +---------------------------+
                 |         User Query         |
                 +-------------+-------------+
                               |
                               v
                 +---------------------------+
                 |   Scope / Safety Guard     |----refuse--> Final Answer (refusal)
                 +-------------+-------------+
                               | passes
                               v
        +----------------------------------------------+
        |        Reasoning Loop (Thought / Act)         |<------------------+
        |  reads/writes  +-----------------------+      |                   |
        |  ------------->|  Memory (working +    |      |                   |
        |                |   session history)    |      |                   |
        |                +-----------------------+      |                   |
        +---------------------+--------------------------+                  |
                              |  Action: call a tool                        |
                              v                                             |
                +--------------------------+   +---------------------+     |
                |     query_dataset()      |   |   safe_calculate()   |     |
                +--------------------------+   +---------------------+     |
                              |  Observation                                |
                              +---------------------------------------------+
                              |  (loop again, or stop at max_steps)
                              v
                 +---------------------------+
                 |       Final Answer         |
                 +---------------------------+
```

**Design choices and trade-offs:**
- **Rule-based intent classification instead of an LLM.** The reasoning step that decides *what kind* of request a query is (lookup, calculation, follow-up, out-of-scope, unclear) is implemented as explicit, inspectable pattern-matching rules rather than a call to a large language model. This was chosen so the whole system runs deterministically and reproducibly with no external API key or network dependency, and so every decision the agent makes is directly traceable to a specific rule in the code. The trade-off is flexibility: an LLM-based version of this same architecture would understand far more varied phrasings of the same question, while this rule-based version will misclassify sufficiently unusual phrasing (demonstrated as an observed limitation in Task 4).
- **Single-agent instead of multi-agent.** A multi-agent design (for example, a separate "retrieval agent" and "calculation agent" coordinated by a planner) was considered but rejected as unnecessary overhead for a task this small; single-agent tool use is sufficient when there are only two tools and no need for specialized sub-agents to negotiate or divide labor.
- **In-memory, session-only state instead of a persistent database.** Memory resets whenever the notebook/session restarts. This keeps the implementation simple and appropriate for a small-scale academic project, at the cost of not remembering anything across separate runs -- a production version would persist memory externally.

## Task 3 — Implementation

### Setup

In [1]:
import ast
import operator
import re
from dataclasses import dataclass, field
from typing import Optional

import pandas as pd

df = pd.read_csv("sales_data.csv")
print(f"Loaded sales_data.csv: {df.shape[0]} rows, {df.shape[1]} columns")
print("Columns:", list(df.columns))
df.head()

Loaded sales_data.csv: 4031 rows, 8 columns
Columns: ['date', 'month', 'product', 'category', 'region', 'quantity', 'unit_price', 'total_sales']


,date,month,product,category,region,quantity,unit_price,total_sales
0,2025-01-01,January,Muffin,Bakery,Uptown,2,3.25,6.50
1,2025-01-01,January,Cold Brew,Beverage,Downtown,3,4.00,12.00
2,2025-01-01,January,Muffin,Bakery,Downtown,1,3.25,3.25
3,2025-01-01,January,Croissant,Bakery,Suburban,3,3.75,11.25
4,2025-01-01,January,Muffin,Bakery,Suburban,3,3.25,9.75


### Tools

`safe_calculate` is deliberately built on Python's `ast` module rather than `eval`/`exec`, so it can only ever parse and evaluate a numeric arithmetic expression -- it is structurally incapable of running arbitrary code, which is the safeguard, not just a comment promising good behavior.

In [2]:
_ALLOWED_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.USub: operator.neg,
    ast.Mod: operator.mod,
}


def safe_calculate(expression: str) -> float:
    """Safely evaluate a basic arithmetic expression (+, -, *, /, %, parentheses)
    using an AST whitelist instead of eval(), so arbitrary code can never run.
    Raises ValueError for anything outside plain arithmetic.
    """
    def _eval(node):
        if isinstance(node, ast.Expression):
            return _eval(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPS:
            return _ALLOWED_OPS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPS:
            return _ALLOWED_OPS[type(node.op)](_eval(node.operand))
        raise ValueError(f"Unsupported or unsafe expression: {ast.dump(node)}")

    tree = ast.parse(expression, mode="eval")
    return _eval(tree)


def query_dataset(data: pd.DataFrame, metric: str, agg: str = "sum",
                   month: Optional[str] = None, product: Optional[str] = None) -> dict:
    """Filter `data` by optional month/product, then aggregate `metric` with `agg`
    (sum, mean, or count). Returns a dict with the result and the filters actually
    applied, or an 'error' key if the metric/filter isn't supported by the schema.
    """
    if metric not in data.columns:
        return {"error": f"Column '{metric}' does not exist in the dataset."}

    filtered = data
    applied = {}
    if month is not None:
        if month not in set(data["month"]):
            return {"error": f"No data for month '{month}'."}
        filtered = filtered[filtered["month"] == month]
        applied["month"] = month
    if product is not None:
        if product not in set(data["product"]):
            return {"error": f"No data for product '{product}'."}
        filtered = filtered[filtered["product"] == product]
        applied["product"] = product

    if filtered.empty:
        return {"error": "No rows matched the given filters."}

    if agg == "sum":
        value = filtered[metric].sum()
    elif agg == "mean":
        value = filtered[metric].mean()
    elif agg == "count":
        value = len(filtered)
    else:
        return {"error": f"Unsupported aggregation '{agg}'."}

    return {"value": round(float(value), 2), "metric": metric, "agg": agg, "filters": applied}


# Quick sanity checks of the tools in isolation
print(safe_calculate("145 * 3 + 20"))
print(query_dataset(df, metric="total_sales", agg="sum", month="March"))

455
{'value': 5200.5, 'metric': 'total_sales', 'agg': 'sum', 'filters': {'month': 'March'}}


### Memory

In [3]:
@dataclass
class AgentMemory:
    """Session-scoped agent memory: a running history of Q&A turns, a cache to
    avoid recomputing repeated questions, and the most recent metric/filters
    the agent resolved, so short follow-up questions can reuse that context.
    """
    history: list = field(default_factory=list)      # [(query, answer), ...]
    cache: dict = field(default_factory=dict)         # normalized_query -> answer
    last_topic: Optional[dict] = None                 # e.g. {"metric": "total_sales", "month": "March"}

    def remember(self, query: str, answer: str, topic: Optional[dict] = None):
        self.history.append((query, answer))
        self.cache[query.strip().lower()] = answer
        if topic is not None:
            self.last_topic = topic

    def check_cache(self, query: str) -> Optional[str]:
        return self.cache.get(query.strip().lower())

### Reasoning Loop, Safeguards, and Agent Class

The agent below implements the Thought/Action/Observation loop described in Task 2. Intent classification, entity extraction (month/product/metric), and the scope guard are all implemented as explicit, commented rules so every decision is traceable to a specific line of code -- this is the "rule-based instead of LLM-based reasoning" design choice discussed above.

In [4]:
MONTHS = ["January", "February", "March", "April", "May", "June"]
PRODUCTS = list(df["product"].unique())
METRIC_KEYWORDS = {
    "total_sales": ["sales", "revenue", "total"],
    "unit_price": ["price", "cost"],
    "quantity": ["quantity", "units", "how many"],
}
DESTRUCTIVE_KEYWORDS = ["delete", "remove", "drop", "update", "edit", "modify", "insert", "overwrite"]
UNSUPPORTED_TOPICS = ["employee", "staff", "profit margin", "forecast", "predict", "next year", "inventory cost"]


class DataDeskAgent:
    def __init__(self, data: pd.DataFrame, max_steps: int = 4):
        self.data = data
        self.memory = AgentMemory()
        self.max_steps = max_steps

    # ---- Reasoning helpers (explicit, inspectable decision rules) ----
    def _extract_month(self, text: str) -> Optional[str]:
        for m in MONTHS:
            if m.lower() in text.lower():
                return m
        return None

    def _extract_product(self, text: str) -> Optional[str]:
        for p in PRODUCTS:
            if p.lower() in text.lower():
                return p
        return None

    def _extract_metric(self, text: str) -> Optional[str]:
        text_l = text.lower()
        for metric, keywords in METRIC_KEYWORDS.items():
            if any(kw in text_l for kw in keywords):
                return metric
        return None

    def _extract_agg(self, text: str) -> str:
        text_l = text.lower()
        if any(w in text_l for w in ["average", "avg", "mean"]):
            return "mean"
        if any(w in text_l for w in ["how many orders", "count", "number of orders"]):
            return "count"
        return "sum"

    def _is_out_of_scope(self, text: str) -> Optional[str]:
        text_l = text.lower()
        if any(kw in text_l for kw in DESTRUCTIVE_KEYWORDS):
            return "That would modify the dataset, which this agent is not permitted to do (read-only scope)."
        if any(kw in text_l for kw in UNSUPPORTED_TOPICS):
            return (f"I don't have that information -- the dataset only tracks "
                    f"{list(self.data.columns)}.")
        return None

    def _is_pure_calculation(self, text: str) -> Optional[str]:
        # Matches queries that are basically just an arithmetic expression,
        # optionally preceded by phrases like "what is" or "calculate".
        cleaned = re.sub(r"(?i)^(what is|what's|calculate|compute)\s*", "", text).strip().rstrip("?")
        if re.fullmatch(r"[0-9\.\s()+\-*/%]+", cleaned) and any(ch.isdigit() for ch in cleaned):
            return cleaned
        return None

    def _looks_like_followup(self, text: str) -> bool:
        text_l = text.lower()
        return any(w in text_l for w in ["that", "compared to", "compare", "what about", "versus", "vs"])

    # ---- Main ReAct-style loop ----
    def run(self, query: str) -> dict:
        trace = [f"Thought: received query -> {query!r}"]

        cached = self.memory.check_cache(query)
        if cached is not None:
            trace.append("Thought: this exact question was already answered this session; reusing cached answer.")
            return {"answer": cached, "trace": trace}

        refusal = self._is_out_of_scope(query)
        if refusal:
            trace.append(f"Action: scope guard triggered. Observation: {refusal}")
            self.memory.remember(query, refusal)
            return {"answer": refusal, "trace": trace}

        steps_used = 0
        while steps_used < self.max_steps:
            steps_used += 1

            calc_expr = self._is_pure_calculation(query)
            if calc_expr:
                trace.append(f"Thought: this looks like a pure calculation ({calc_expr!r}).")
                try:
                    result = safe_calculate(calc_expr)
                    trace.append(f"Action: safe_calculate({calc_expr!r}). Observation: {result}")
                    answer = f"{result}"
                    self.memory.remember(query, answer)
                    return {"answer": answer, "trace": trace}
                except Exception as exc:
                    trace.append(f"Observation: calculation failed ({exc}).")
                    answer = "I couldn't evaluate that as a plain arithmetic expression."
                    self.memory.remember(query, answer)
                    return {"answer": answer, "trace": trace}

            month = self._extract_month(query)
            product = self._extract_product(query)
            metric = self._extract_metric(query)
            agg = self._extract_agg(query)

            if metric is None and self._looks_like_followup(query) and self.memory.last_topic:
                trace.append("Thought: no metric mentioned, but this looks like a follow-up; reusing last topic from memory.")
                metric = self.memory.last_topic.get("metric")
                product = product or self.memory.last_topic.get("product")
                # A comparison follow-up ("compared to February") usually supplies a *new* month
                # while keeping the same metric/product as before.

            if metric is None:
                trace.append("Observation: could not identify a metric to look up, and no usable memory context.")
                answer = ("I'm not sure what you're asking for. Could you mention a metric "
                          "such as sales, price, or quantity?")
                self.memory.remember(query, answer)
                return {"answer": answer, "trace": trace}

            trace.append(f"Thought: interpreted as a lookup -> metric={metric}, agg={agg}, month={month}, product={product}.")
            result = query_dataset(self.data, metric=metric, agg=agg, month=month, product=product)
            trace.append(f"Action: query_dataset(metric={metric!r}, agg={agg!r}, month={month!r}, product={product!r}). "
                          f"Observation: {result}")

            if "error" in result:
                answer = f"I couldn't answer that: {result['error']}"
                self.memory.remember(query, answer)
                return {"answer": answer, "trace": trace}

            filt_desc = ", ".join(f"{k}={v}" for k, v in result["filters"].items()) or "all data"
            answer = f"The {agg} of {metric} ({filt_desc}) is {result['value']}."
            self.memory.remember(query, answer, topic={"metric": metric, "month": month, "product": product})
            return {"answer": answer, "trace": trace}

        trace.append(f"Observation: reached the maximum of {self.max_steps} reasoning steps without a resolved answer.")
        answer = "I wasn't able to resolve this request within my step limit."
        self.memory.remember(query, answer)
        return {"answer": answer, "trace": trace}


agent = DataDeskAgent(df)
print("Agent initialized with", len(df), "rows and max_steps =", agent.max_steps)

Agent initialized with 4031 rows and max_steps = 4


## Task 4 — Execute and Observe Agent Behavior

The scenarios below are run in a single session (sharing one `agent` instance and its memory) to also exercise the follow-up and caching behavior described in Task 2.

In [5]:
def run_and_show(query: str):
    result = agent.run(query)
    print(f"Q: {query}")
    print(f"A: {result['answer']}")
    for step in result["trace"]:
        print("   ", step)
    print()
    return result


_ = run_and_show("What were total sales in March?")
_ = run_and_show("What's the average unit price for Espresso?")
_ = run_and_show("How does that compare to February?")
_ = run_and_show("What's 145 * 3 + 20?")
_ = run_and_show("Can you delete all the sales records?")
_ = run_and_show("What is the profit margin by employee?")

Q: What were total sales in March?
A: The sum of total_sales (month=March) is 5200.5.
    Thought: received query -> 'What were total sales in March?'
    Thought: interpreted as a lookup -> metric=total_sales, agg=sum, month=March, product=None.
    Action: query_dataset(metric='total_sales', agg='sum', month='March', product=None). Observation: {'value': 5200.5, 'metric': 'total_sales', 'agg': 'sum', 'filters': {'month': 'March'}}

Q: What's the average unit price for Espresso?
A: The mean of unit_price (product=Espresso) is 3.25.
    Thought: received query -> "What's the average unit price for Espresso?"
    Thought: interpreted as a lookup -> metric=unit_price, agg=mean, month=None, product=Espresso.
    Action: query_dataset(metric='unit_price', agg='mean', month=None, product='Espresso'). Observation: {'value': 3.25, 'metric': 'unit_price', 'agg': 'mean', 'filters': {'product': 'Espresso'}}

Q: How does that compare to February?
A: The sum of unit_price (month=February, product=

**Notes on how the agent reasoned through these six scenarios:**
- **Query 1** correctly extracted `total_sales` from the word "sales" and `March` from the text, called `query_dataset`, and returned the observed value.
- **Query 2** correctly detected "average" -> `agg="mean"`, "price" -> `metric="unit_price"`, and "Espresso" -> `product="Espresso"`.
- **Query 3** is a genuine failure case, discovered by actually running the agent rather than designed in advance -- see the dedicated discussion below.
- **Query 4** correctly recognized a pure arithmetic expression and routed to `safe_calculate` instead of the dataset tool at all, skipping the dataset-lookup branch entirely.
- **Query 5** triggered the scope guard's destructive-keyword check ("delete") before any tool ran, and refused.
- **Query 6** triggered the scope guard's unsupported-topic check ("employee") and reported exactly which columns are available instead of guessing a number.

**Failure case (discovered during testing, not designed in advance):** Look closely at Query 3's trace. Query 2 asked for the *average* (`mean`) unit price of Espresso, which correctly set `last_topic = {"metric": "unit_price", ...}`. Query 3, "How does that compare to February?", correctly reused `metric="unit_price"` and `product="Espresso"` from memory -- but the aggregation type (`mean`) was **not** carried over, because `_extract_agg` always re-derives `agg` from the new query's own text, and "How does that compare to February?" contains no aggregation keyword, so it silently defaulted back to `"sum"`. The agent answered with the *sum* of unit prices (351.0, a number with no real-world meaning) when the user almost certainly wanted the *average* price in February to compare against the average price it had just reported. This is a genuine bug in the follow-up-resolution logic: it correctly remembers *what* the previous question was about but not *how* it was being aggregated. A fix would be to add `agg` to `last_topic` and prefer it over the default whenever the current query doesn't explicitly state a different aggregation.

In [6]:
# A second, independently discovered limitation: no understanding of relative time references
_ = run_and_show("What was the average price of a Latte last month?")

Q: What was the average price of a Latte last month?
A: The mean of unit_price (product=Latte) is 4.5.
    Thought: received query -> 'What was the average price of a Latte last month?'
    Thought: interpreted as a lookup -> metric=unit_price, agg=mean, month=None, product=Latte.
    Action: query_dataset(metric='unit_price', agg='mean', month=None, product='Latte'). Observation: {'value': 4.5, 'metric': 'unit_price', 'agg': 'mean', 'filters': {'product': 'Latte'}}



**Second limitation:** `_extract_month` only recognizes literal calendar month names (January-June); it has no concept of "last month," "this month," or any other relative time reference, and there is no current-date context anywhere in the system for it to resolve one against. Rather than asking for clarification, the agent silently drops the unrecognized time reference and answers the question as if "last month" had not been said at all -- averaging over every month in the dataset instead of one specific month. This is arguably worse than an explicit refusal, since the answer looks confident and complete while quietly answering a broader question than the one asked. A production version of this agent would need either an explicit "current date" concept to resolve relative time phrases, or a safeguard that detects unrecognized time language and asks the user to name a specific month rather than silently ignoring it.

In [7]:
# Demonstrate the answer cache: asking the exact same question again should short-circuit
# straight to the cached answer without re-invoking any tool.
_ = run_and_show("What were total sales in March?")

print(f"Session history length: {len(agent.memory.history)} question/answer pairs recorded.")

Q: What were total sales in March?
A: The sum of total_sales (month=March) is 5200.5.
    Thought: received query -> 'What were total sales in March?'
    Thought: this exact question was already answered this session; reusing cached answer.

Session history length: 7 question/answer pairs recorded.
